In [1]:
# =========================================================
# TVP-VAR Connectedness Baseline
# - Recursive least squares with forgetting factor
# - Time-varying VAR coefficients
# - Generalized FEVD
# - Outputs: total / TO / FROM / NET spillover series
# =========================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import inv, pinv
from scipy.linalg import cholesky

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "./spillover_data.csv"
OUT_DIR = "./output_tvp_var"

DATE_COL = "Date"
VARS = ["dlog_SOLVPN", "dlog_COPPER", "dlog_DXY", "d_UST10Y", "dlog_VIX"]

P = 1
H = 10                      # FEVD horizon
LAMBDA_BETA = 0.99          # forgetting factor for coefficients
LAMBDA_SIGMA = 0.98         # EWMA covariance update
MIN_T = 150                 # warm-up for stable estimation

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[[DATE_COL] + VARS].dropna().reset_index(drop=True)

Y = df[VARS].copy()
dates = df[DATE_COL].copy()

T = len(Y)
K = len(VARS)

print("Data shape:", Y.shape)

# -----------------------------
# Helper functions
# -----------------------------
def make_var_lagged(df_y: pd.DataFrame, p: int):
    X_list = []
    colnames = []
    for lag in range(1, p + 1):
        lagged = df_y.shift(lag)
        lagged.columns = [f"{c}_L{lag}" for c in df_y.columns]
        X_list.append(lagged)
        colnames.extend(lagged.columns.tolist())
    X = pd.concat(X_list, axis=1)
    out = pd.concat([df_y, X], axis=1).dropna().reset_index(drop=True)
    Y_mat = out[df_y.columns].values
    X_mat = out[colnames].values
    return Y_mat, X_mat, out.index, colnames

def build_companion(beta_flat: np.ndarray, k: int, p: int):
    """
    beta_flat shape: (k*p,) for one equation excluding intercept
    Here we reconstruct A1...Ap only for one equation externally.
    """
    raise NotImplementedError

def var_companion(A_list):
    """
    A_list: list of A_l matrices, each shape (k, k)
    returns companion matrix of shape (k*p, k*p)
    """
    p = len(A_list)
    k = A_list[0].shape[0]
    top = np.hstack(A_list)
    if p == 1:
        return top
    bottom = np.hstack([np.eye(k*(p-1)), np.zeros((k*(p-1), k))])
    F = np.vstack([top, bottom])
    return F

def generalized_fevd(A_list, Sigma, H):
    """
    Pesaran-Shin generalized FEVD for VAR(p)
    A_list: [A1, ..., Ap], each (k,k)
    Sigma: residual covariance (k,k)
    Returns theta normalized row-wise (k,k)
    """
    k = Sigma.shape[0]
    p = len(A_list)

    F = var_companion(A_list)
    if p == 1:
        kp = k
    else:
        kp = k * p

    J = np.hstack([np.eye(k), np.zeros((k, kp - k))]) if p > 1 else np.eye(k)

    Phi = []
    F_power = np.eye(kp)
    for h in range(H):
        Phi_h = J @ F_power @ J.T
        Phi.append(Phi_h)
        F_power = F_power @ F

    theta = np.zeros((k, k))
    sigma_diag = np.diag(Sigma).copy()
    sigma_diag[sigma_diag <= 1e-12] = 1e-12

    for i in range(k):
        for j in range(k):
            num = 0.0
            den = 0.0
            e_i = np.zeros((k, 1)); e_i[i, 0] = 1.0
            e_j = np.zeros((k, 1)); e_j[j, 0] = 1.0
            for h in range(H):
                phi_h = Phi[h]
                num += float((e_i.T @ phi_h @ Sigma @ e_j) ** 2 / sigma_diag[j])
                den += float(e_i.T @ phi_h @ Sigma @ phi_h.T @ e_i)
            theta[i, j] = num / max(den, 1e-12)

    row_sums = theta.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 1e-12] = 1e-12
    theta_norm = theta / row_sums
    return theta_norm

def spillover_from_fevd(theta):
    """
    theta: row-normalized FEVD matrix
    """
    k = theta.shape[0]
    off_diag_sum = theta.sum() - np.trace(theta)
    total = 100.0 * off_diag_sum / k

    to_others = 100.0 * (theta.sum(axis=0) - np.diag(theta))
    from_others = 100.0 * (theta.sum(axis=1) - np.diag(theta))
    net = to_others - from_others

    return total, to_others, from_others, net

# -----------------------------
# Prepare lagged matrices
# -----------------------------
Y_mat, X_lag_only, valid_idx, lag_cols = make_var_lagged(Y, P)
dates2 = dates.iloc[valid_idx].reset_index(drop=True)

# Add intercept
X_full = np.column_stack([np.ones(len(X_lag_only)), X_lag_only])  # (T-p, 1+k*p)
Y_full = Y_mat.copy()

T2 = len(Y_full)
n_params = X_full.shape[1]

# -----------------------------
# TVP-RLS estimation
# Equation-by-equation
# -----------------------------
betas_t = np.zeros((T2, K, n_params))
resids_t = np.zeros((T2, K))

# Initialize OLS on initial window
init_n = max(MIN_T, n_params + 20)
X_init = X_full[:init_n]
Y_init = Y_full[:init_n]

for eq in range(K):
    beta0 = pinv(X_init.T @ X_init) @ (X_init.T @ Y_init[:, eq])
    P0 = np.eye(n_params) * 1000.0

    beta_prev = beta0.copy()
    P_prev = P0.copy()

    for t in range(T2):
        x_t = X_full[t:t+1].T
        y_t = Y_full[t, eq]

        # Predict
        y_hat = float(beta_prev @ x_t[:, 0])
        err = y_t - y_hat

        # RLS update
        denom = LAMBDA_BETA + float(x_t.T @ P_prev @ x_t)
        K_t = (P_prev @ x_t) / max(denom, 1e-12)
        beta_new = beta_prev + (K_t[:, 0] * err)
        P_new = (P_prev - K_t @ x_t.T @ P_prev) / LAMBDA_BETA

        betas_t[t, eq, :] = beta_new
        resids_t[t, eq] = err

        beta_prev = beta_new
        P_prev = P_new

# -----------------------------
# Time-varying Sigma via EWMA
# -----------------------------
Sigma_t = np.zeros((T2, K, K))
Sigma_prev = np.cov(resids_t[:init_n].T)

for t in range(T2):
    e_t = resids_t[t].reshape(-1, 1)
    Sigma_new = LAMBDA_SIGMA * Sigma_prev + (1 - LAMBDA_SIGMA) * (e_t @ e_t.T)

    # numerical stabilization
    Sigma_new = (Sigma_new + Sigma_new.T) / 2
    min_diag = 1e-8
    d = np.diag(Sigma_new).copy()
    d[d < min_diag] = min_diag
    np.fill_diagonal(Sigma_new, d)

    Sigma_t[t] = Sigma_new
    Sigma_prev = Sigma_new

# -----------------------------
# FEVD / Connectedness over time
# -----------------------------
records = []
pairwise_records = []

for t in range(T2):
    if t < MIN_T:
        continue

    # Build A1...Ap from time-varying betas
    # betas_t[t, eq, :] = [intercept, lag coefficients...]
    # for p=1, lag coefficient vector length = K
    A_list = []
    coef_block = betas_t[t, :, 1:]  # shape (K, K*P)

    for lag in range(P):
        A_l = coef_block[:, lag*K:(lag+1)*K]
        A_list.append(A_l)

    Sigma = Sigma_t[t]
    theta = generalized_fevd(A_list, Sigma, H)
    total, to_, from_, net_ = spillover_from_fevd(theta)

    row = {
        "Date": dates2.iloc[t],
        "Total_Spillover": total
    }

    for i, v in enumerate(VARS):
        row[f"{v}_TO"] = to_[i]
        row[f"{v}_FROM"] = from_[i]
        row[f"{v}_NET"] = net_[i]

    records.append(row)

    for i, vi in enumerate(VARS):
        for j, vj in enumerate(VARS):
            pairwise_records.append({
                "Date": dates2.iloc[t],
                "Response": vi,
                "Shock": vj,
                "FEVD": theta[i, j]
            })

tvp_conn = pd.DataFrame(records)
tvp_pairwise = pd.DataFrame(pairwise_records)

# -----------------------------
# Save outputs
# -----------------------------
tvp_conn.to_csv(os.path.join(OUT_DIR, "tvp_var_connectedness.csv"), index=False)
tvp_pairwise.to_csv(os.path.join(OUT_DIR, "tvp_var_pairwise_fevd.csv"), index=False)

# average FEVD matrix
avg_mat = (
    tvp_pairwise.groupby(["Response", "Shock"])["FEVD"]
    .mean()
    .unstack()
    .reindex(index=VARS, columns=VARS)
)
avg_mat.to_csv(os.path.join(OUT_DIR, "tvp_var_avg_fevd_matrix.csv"))

# metadata
meta = {
    "model": "TVP-VAR Connectedness (RLS approximation)",
    "vars": VARS,
    "lag_order": P,
    "fevd_horizon": H,
    "lambda_beta": LAMBDA_BETA,
    "lambda_sigma": LAMBDA_SIGMA,
    "min_t": MIN_T,
    "n_obs_used": int(T2)
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(tvp_conn["Date"]), tvp_conn["Total_Spillover"])
plt.title("TVP-VAR Total Spillover")
plt.xlabel("Date")
plt.ylabel("Connectedness (%)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tvp_var_total_spillover.png"), dpi=300)
plt.show()

plt.figure(figsize=(12, 5))
for v in ["dlog_SOLVPN", "dlog_COPPER"]:
    plt.plot(pd.to_datetime(tvp_conn["Date"]), tvp_conn[f"{v}_NET"], label=v)
plt.title("TVP-VAR Net Spillover")
plt.xlabel("Date")
plt.ylabel("NET")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tvp_var_net_spillover_keyvars.png"), dpi=300)
plt.show()

print("Saved to:", OUT_DIR)
print(tvp_conn.tail())

FileNotFoundError: [Errno 2] No such file or directory: './spillover_data.csv'